# Pull a year of campaign data

*Data Analysis in Python · Lesson 12 final exercise*

Your media team wants 2025's return on ad spend by month, in a Google Sheet they can open. Pull the year from the Simmer API, summarize it by month and share it.

**Before you start**, choose **File → Save a copy in Drive**, and work in the copy that opens. This workbook is shared with every student, so Colab doesn't save your changes to it.

Add your cells between the login cell and the **Check your work** cell. If you're stuck on a step, open its hint under the **Check your work** cell.

1. Run the parameters cell and the login cell. The parameters cell reads the course API key from Colab Secrets, as in topic 12.1. For the login, choose the Google account you use for Colab, and **Allow**.
2. Write the function `get_json(url, params, tries=5)` from topic 12.3., which waits and tries again after a 429 or a 503.
3. Page through the `reports/daily` endpoint for `START_DATE` to `END_DATE`, with `campaign` as the dimension and `PAGE_SIZE` rows per page, using `get_json()`. Create the `report` table from the rows, and `row_count`: the `row_count` the API reported.
4. Create the `monthly` table: for the rows with spend, one row per month, with these columns:
   - `month`: the month as text, such as `2025-01`
   - `spend` and `revenue`: their sums
   - `roas`: revenue divided by spend
5. Write the `monthly` table to the spreadsheet called `SHEET_NAME`, opened if it exists and created if it doesn't, as in topic 12.5. Then create the `read_back` table: every row of the sheet, read back.
6. Choose **Runtime → Restart session and run all**, and read the **Check your work** cell's output.

**Expected output**: The `report` table has 8,139 rows, the same as `row_count`, with $318,705.08 of spend. The `monthly` table has 12 months: November's ROAS is about `2.867`. The `read_back` table has the same 12 months, read back from the sheet.

## Parameters

In [ ]:
import time

import pandas as pd
import requests
from google.colab import userdata

BASE_URL = "https://api.teamsimmer.com/v1/"
API_KEY = userdata.get("SIMMER_API_KEY")  # Colab Secrets, as in topic 12.1
HEADERS = {"X-API-Key": API_KEY}
START_DATE, END_DATE = "2025-01-01", "2025-12-31"
PAGE_SIZE = 500
SHEET_NAME = "Simmer 2025 campaign report"

In [ ]:
# Log in to Google as yourself. Run it, choose your account and Allow.
import google.auth
import gspread
from google.colab import auth

auth.authenticate_user()
credentials, _ = google.auth.default()
gc = gspread.authorize(credentials)

In [ ]:
RETRY_STATUSES = {429, 503}


def get_json(url, params, tries=5):
    """Return the JSON body of a GET request, waiting and trying again after a 429 or 503."""
    for attempt in range(tries):
        response = requests.get(url, headers=HEADERS, params=params)
        if response.status_code not in RETRY_STATUSES:
            response.raise_for_status()
            return response.json()
        wait = int(response.headers.get("Retry-After", 1))
        print(f"{response.status_code}, waiting {wait} s")
        time.sleep(wait)
    raise RuntimeError(f"Still {response.status_code} after {tries} tries: {url}")

In [ ]:
params = {"start_date": START_DATE, "end_date": END_DATE, "dimensions": "campaign", "page_size": PAGE_SIZE}
rows = []
while True:
    body = get_json(BASE_URL + "reports/daily", params)
    rows.extend(body["rows"])
    if "next_page_token" not in body:
        break
    params["page_token"] = body["next_page_token"]
report = pd.DataFrame(rows)
row_count = body["row_count"]
len(report), row_count

In [ ]:
paid = report[report["spend"] > 0].copy()
paid["month"] = pd.to_datetime(paid["date"]).dt.strftime("%Y-%m")
monthly = paid.groupby("month", as_index=False).agg(spend=("spend", "sum"), revenue=("revenue", "sum"))
monthly["roas"] = monthly["revenue"] / monthly["spend"]
monthly.round(3)

In [ ]:
try:
    spreadsheet = gc.open(SHEET_NAME)
except gspread.SpreadsheetNotFound:
    spreadsheet = gc.create(SHEET_NAME)
worksheet = spreadsheet.sheet1
worksheet.clear()
worksheet.update([monthly.columns.tolist()] + monthly.values.tolist())
read_back = pd.DataFrame(worksheet.get_all_records())
read_back, spreadsheet.url

In [ ]:
# @title Check your work
# @markdown This cell runs last, with the others. Run it with Runtime → Restart session and run all.
# This cell only reads your variables and the cells that have run. It doesn't change anything.
def check_your_work():
    import sys
    import pandas as pd
    found = globals()
    results = []

    inputs = found.get("In")
    history = [source for source in inputs[1:-1] if "def check_your_work" not in source] if isinstance(inputs, list) else None
    checked_before = isinstance(inputs, list) and any("def check_your_work" in source for source in inputs[1:-1])
    if history is None:
        results.append((None, "This check can't see which cells have run, so it can't tell whether this was a clean run. Choose Runtime → Restart session and run all to be sure."))
    else:
        repeated = [source for source in history if source.strip() and history.count(source) > 1]
        error = getattr(sys, "last_exc", None) or getattr(sys, "last_value", None)
        if checked_before:
            results.append((None, "This cell has run before in this session, so the cells above may have changed since they ran. Choose Runtime → Restart session and run all to check the notebook as it is now."))
        elif repeated:
            first_line = repeated[0].strip().splitlines()[0]
            results.append((None, f"Some cells ran more than once in this session, such as the one that starts with {first_line!r}. Choose Runtime → Restart session and run all, so every cell runs once, from the top."))
        elif error is not None and "429" in str(error):
            results.append((None, "A request stopped with 429 Too Many Requests. Use get_json() for every request, so it waits and tries again, then choose Runtime → Restart session and run all."))
        elif error is not None:
            results.append((None, f"A cell stopped with {type(error).__name__} earlier in this session, so the cells after it may not have run in order. Choose Runtime → Restart session and run all."))
        else:
            results.append((True, "Every cell in this session ran once, with no errors."))

    if not callable(found.get("get_json")):
        results.append((False, "get_json should be a function: topic 12.3.'s def get_json(url, params, tries=5): …"))

    report, row_count = found.get("report"), found.get("row_count")
    if not isinstance(report, pd.DataFrame) or "spend" not in report.columns:
        results.append((False, "report should be a DataFrame of every page's rows, with the spend column."))
    elif "campaign" not in report.columns:
        results.append((False, "The report table has no campaign column. Ask for dimensions=\"campaign\"."))
    elif row_count != 8139:
        results.append((False, f"row_count should be the row_count the API reported: body[\"row_count\"], 8,139. It is {row_count!r}."))
    elif len(report) == 500:
        results.append((False, "The report table has only the first page: 500 rows. Page through every page, as in topic 12.2."))
    elif len(report) != row_count:
        results.append((False, f"The report table has {len(report):,} rows, and the API reported {row_count:,}. Page through every page."))
    elif abs(report["spend"].sum() - 318705.08) > 0.01:
        results.append((False, f"The report table should have $318,705.08 of spend, for all of 2025. It has ${report['spend'].sum():,.2f}."))
    else:
        results.append((True, "8,139 rows of 2025 by campaign, as the API reported: $318,705.08 of spend."))

    monthly = found.get("monthly")
    columns = ["month", "spend", "revenue", "roas"]
    if not isinstance(monthly, pd.DataFrame) or [column for column in columns if column not in monthly.columns]:
        results.append((False, "monthly should be a DataFrame with the columns month, spend, revenue and roas. Use groupby(\"month\", as_index=False), so month is a column."))
    elif len(monthly) != 12:
        results.append((False, f"The monthly table should have the 12 months of 2025. It has {len(monthly)} rows. Make month the month as text: .dt.strftime(\"%Y-%m\")."))
    else:
        november = monthly[monthly["month"].astype(str) == "2025-11"]
        roas = float(november["roas"].iloc[0]) if len(november) else None
        if roas is None:
            results.append((False, f"The monthly table should have months as text, such as 2025-11. It has {list(monthly['month'])[:2]}."))
        elif abs(roas - 13.666) < 0.01:
            results.append((False, "November's ROAS is 13.67: the revenue includes the rows without spend, such as organic and email. Keep the rows with spend > 0 before you sum."))
        elif abs(roas - 0.3488) < 0.001:
            results.append((False, "November's ROAS is 0.35: that's spend divided by revenue. ROAS is revenue divided by spend."))
        elif abs(roas - 2.8671) > 0.001 or abs(monthly["spend"].sum() - 318705.08) > 0.01:
            results.append((False, f"November's ROAS should be about 2.867, and the year's spend $318,705.08. They are {roas!r} and ${monthly['spend'].sum():,.2f}."))
        else:
            results.append((True, "12 months of ROAS, from 2.19 in August to 2.87 in November."))

    read_back = found.get("read_back")
    if not isinstance(read_back, pd.DataFrame):
        results.append((False, "read_back should be a DataFrame: pd.DataFrame(worksheet.get_all_records())."))
    elif [column for column in columns if column not in read_back.columns]:
        results.append((False, f"The sheet's header row should be the monthly table's columns. read_back has {list(read_back.columns)}."))
    elif len(read_back) != 12:
        results.append((False, f"The sheet should have 12 months below its header. read_back has {len(read_back)} rows. Clear the worksheet before you write."))
    elif abs(read_back["spend"].sum() - 318705.08) > 0.01:
        results.append((False, f"The sheet's spend should add up to $318,705.08, as the monthly table's does. It adds up to ${read_back['spend'].sum():,.2f}."))
    else:
        results.append((True, "The sheet reads back as the monthly table: 12 months and $318,705.08 of spend."))

    for passed, message in results:
        print({True: "✅ ", False: "❌ ", None: "⚠️ "}[passed] + message)
    if any(passed is False for passed, message in results):
        print("\nFix the first ❌, then choose Runtime → Restart session and run all.")
    elif any(passed is None for passed, message in results):
        print("\nYour results match. Restart the session and run all, to check them in a clean run.")
    else:
        print("\nEverything matches, in a clean run. Open spreadsheet.url: that's what your media team will see.")


check_your_work()

## Hints

Stuck on a step? Open its hint. Each one says where the course showed it, and gives the shape of the code, with `...` where your part goes. Try the step from the hint before you open the solution.

<details>
<summary><b>Step 2: the retry function</b></summary>

Topic 12.3.: retry only the statuses that are temporary, wait as long as `Retry-After` says, and give up after `tries` tries.

~~~python
RETRY_STATUSES = {429, 503}


def get_json(url, params, tries=5):
    """Return the JSON body of a GET request, waiting and trying again after a 429 or 503."""
    for attempt in range(tries):
        response = requests.get(url, headers=..., params=...)
        if response.status_code not in RETRY_STATUSES:
            response.raise_for_status()
            return ...
        time.sleep(int(response.headers.get("Retry-After", 1)))
    raise RuntimeError(...)
~~~

</details>

<details>
<summary><b>Step 3: every page</b></summary>

Topic 12.2.'s loop, with `get_json()` for each request, as in topic 12.3.

~~~python
params = {"start_date": ..., "end_date": ..., "dimensions": ..., "page_size": ...}
rows = []
while True:
    body = get_json(BASE_URL + "reports/daily", params)
    rows.extend(...)
    if "next_page_token" not in body:
        break
    params["page_token"] = ...
report = pd.DataFrame(rows)
row_count = body["row_count"]
~~~

</details>

<details>
<summary><b>Step 4: the monthly table</b></summary>

Keep the rows with spend, turn each date into its month as text with topic 12.5.'s `strftime()`, and sum by month with lesson 4.'s named aggregation.

~~~python
paid = report[report["spend"] > 0].copy()
paid["month"] = pd.to_datetime(paid["date"]).dt.strftime(...)
monthly = paid.groupby("month", as_index=False).agg(spend=(..., "sum"), revenue=(..., "sum"))
monthly["roas"] = ...
~~~

</details>

<details>
<summary><b>Step 5: the sheet</b></summary>

Topic 12.5.: open the spreadsheet or create it, clear the first tab, write the header and the rows, and read them back.

~~~python
try:
    spreadsheet = gc.open(SHEET_NAME)
except gspread.SpreadsheetNotFound:
    spreadsheet = gc.create(SHEET_NAME)
worksheet = spreadsheet.sheet1
worksheet.clear()
worksheet.update([monthly.columns.tolist()] + ...)
read_back = pd.DataFrame(worksheet.get_all_records())
~~~

</details>

<details>
<summary><b>Show solution</b></summary>

Try the exercise yourself first. This is one way to solve it: if the check passes, your way is right too.

```python
RETRY_STATUSES = {429, 503}


def get_json(url, params, tries=5):
    """Return the JSON body of a GET request, waiting and trying again after a 429 or 503."""
    for attempt in range(tries):
        response = requests.get(url, headers=HEADERS, params=params)
        if response.status_code not in RETRY_STATUSES:
            response.raise_for_status()
            return response.json()
        wait = int(response.headers.get("Retry-After", 1))
        print(f"{response.status_code}, waiting {wait} s")
        time.sleep(wait)
    raise RuntimeError(f"Still {response.status_code} after {tries} tries: {url}")
```

```python
params = {"start_date": START_DATE, "end_date": END_DATE, "dimensions": "campaign", "page_size": PAGE_SIZE}
rows = []
while True:
    body = get_json(BASE_URL + "reports/daily", params)
    rows.extend(body["rows"])
    if "next_page_token" not in body:
        break
    params["page_token"] = body["next_page_token"]
report = pd.DataFrame(rows)
row_count = body["row_count"]
len(report), row_count
```

```python
paid = report[report["spend"] > 0].copy()
paid["month"] = pd.to_datetime(paid["date"]).dt.strftime("%Y-%m")
monthly = paid.groupby("month", as_index=False).agg(spend=("spend", "sum"), revenue=("revenue", "sum"))
monthly["roas"] = monthly["revenue"] / monthly["spend"]
monthly.round(3)
```

```python
try:
    spreadsheet = gc.open(SHEET_NAME)
except gspread.SpreadsheetNotFound:
    spreadsheet = gc.create(SHEET_NAME)
worksheet = spreadsheet.sheet1
worksheet.clear()
worksheet.update([monthly.columns.tolist()] + monthly.values.tolist())
read_back = pd.DataFrame(worksheet.get_all_records())
read_back, spreadsheet.url
```

</details>